In [10]:
import os
import pandas as pd
from IPython.display import Markdown, display
import re

In [11]:
def stitch_markdowns(
        parsed_dir: str = "/Users/Advay/Desktop/Relativity/users/advay/qg/parsed",
) -> str:

    markdown_files = sorted(
        os.path.join(parsed_dir, file_name)
        for file_name in os.listdir(parsed_dir)
        if file_name.endswith(".md")
    )

    stitched = "\n\n".join(
        open(file_path, "r", encoding="utf-8").read().strip()
        for file_path in markdown_files
    )

    return stitched


def parse_stiched(stitched: str) -> pd.DataFrame:
    
    pattern = re.compile(
        r"^##\s*(?P<question_number>\d+)\s+(?P<name>.+?)\n"
        r"\s*Company\s*:\s*(?P<companies>[^\n]*)\n"
        r"\s*##\s*Prompt:\s*\n(?P<prompt>.*?)"
        r"\s*##\s*Explanation:\s*\n(?P<explanation>.*?)"
        r"(?=^\s*##\s*\d+\s+|\Z)",
        re.DOTALL | re.MULTILINE
    )

    rows = []
    for m in pattern.finditer(stitched):
        companies_raw = m.group("companies").strip()

        if companies_raw:
            companies = [
                c.strip()
                for c in companies_raw.split(",")
                if c.strip() and c.strip() != "None"
            ]
        else:
            companies = []
        
        rows.append({
            "index": int(m.group("question_number")),
            "name": m.group("name").strip(),
            "companies": companies,
            "prompt": m.group("prompt").strip(),
            "explanation": m.group("explanation").strip(),
        })

    return pd.DataFrame(rows).set_index("index")

def explode_column(df: pd.DataFrame, column: str = 'companies') -> pd.DataFrame:
    
    return (
        df.explode(column)
        .rename(columns={'name': 'question', column: column})
        .dropna(subset=[column])
        .reset_index(drop=True)
    )[[column, 'question']]

In [19]:
stitched = stitch_markdowns()
df = parse_stiched(stitched)
df

,name,companies,prompt,explanation
index,,,,
1,Place or Take,[Jane Street],You are playing a one-player game with two opa...,There are two aspects to this game that should...
2,Collecting Toys II,"[Optiver, Squarepoint Capital]",Every box of cereal contains one toy from a gr...,Let X be the number of distinct toys you colle...
3,Chess Tournament I,[Arrowstreet Capital],"A chess tournament has 128 players, each with ...",The highest rated player ( P 1 ) will always m...
4,Poker Hands I,[],A poker hand consists of five cards from a sta...,There are a total of ( 52 5 ) total hand combi...
5,Free Sundae,[],You are in line at an ice cream shop when the ...,You choose to be the n -th person in line. In ...
...,...,...,...,...
1135,Choose Your Profit,[],You play a game where you initially start with...,We want our expected payout to be constant no ...
1136,12 Left,[],"Three players, say A,B, and C , are going to d...",There are 7 even values from 1 -15 . For B to ...
1137,Equal Unequal Game,[Squarepoint Capital],Abby and Ben are playing a coin flipping game....,Let A be the event that Abby wins the game. Ab...


In [20]:
all_nums = set(range(1, 1139+1))
index_nums = set(df.index.to_list())
all_nums - index_nums

set()

In [21]:
explode_column(df)['companies'].value_counts()

companies
SIG                                                                                                                                   126
Jane Street                                                                                                                           119
Citadel                                                                                                                                76
Five Rings                                                                                                                             44
Akuna                                                                                                                                  35
WorldQuant                                                                                                                             33
Old Mission                                                                                                                            31
Two Sigma               